# FLChain Last-Layer Compression, Gradient Geometry, and Year-Input Sensitivity Probe (Work 6b)

This canonical notebook evaluates DR-TabPFN (`single_fast`, 1995-1997 trailing context) on the Mayo Clinic FLChain cohort (`7,874` subjects, `7` dense clinical features, native `creatinine` missingness). It tests whether last-layer embedding compression, 1-D head-gradient alignment, random-direction TCAV saturation, and fixed-patient `sample.yr` predictive-entropy sensitivity replicate outside the sparse 500-feature SUS CKD setting.

- **Primary target:** uncensored 3-year fixed-horizon mortality (`H = 1095` days).
- **Sensitivity target:** uncensored 5-year fixed-horizon mortality (`H = 1825` days).
- **Secondary target (geometry + AUROC/Brier only):** raw `death` (excluded from fixed-patient `sample.yr` entropy sensitivity because follow-up duration varies by enrollment year).
- **Temporal scaling:** consistent context-based scaling (`ContextMinMaxScaler`, primary) and historical cached scaling (sensitivity).

In [1]:
import gc
import hashlib
import json
import os
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import tabpfn
import torch

import flchain_probe as probe
import tcav_geometry as geometry

T0 = time.monotonic()
STARTED_UTC = datetime.now(timezone.utc).isoformat()
CSV_PATH = Path("data/flchain/free_light_chain_mortality.csv")
OUT_DIR = Path(os.environ.get("RPI_OUT_DIR", "stats/flchain_last_layer_probe"))
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else None
print(f"Device: {DEVICE} ({GPU_NAME}) | Out: {OUT_DIR} | Torch: {torch.__version__} | sklearn: {sklearn.__version__}")

Device: cuda (NVIDIA GeForce RTX 2080) | Out: /home/rafaelgomes/SAE-TCAV/stats/flchain_last_layer_probe | Torch: 2.1.2+cu121 | sklearn: 1.5.2


## 1. Dataset Integrity & Censoring Audit
Verify the CSV SHA-256 digest and quantify right-censoring across enrollment years for 3-year (`1095d`), 5-year (`1825d`), and 7-year (`2555d`) horizons alongside raw `death`.

In [2]:
checks = []

def record_check(check_name, target, scaling, passed, metric, detail):
    row = {
        "check_name": str(check_name),
        "target": str(target),
        "scaling": str(scaling),
        "passed": bool(passed),
        "metric": float(metric) if metric is not None and np.isfinite(metric) else np.nan,
        "detail": str(detail),
    }
    checks.append(row)

observed_csv_sha = probe.file_sha256(CSV_PATH)
record_check(
    "csv_sha256", "all", "none",
    observed_csv_sha == probe.EXPECTED_CSV_SHA256, np.nan,
    f"observed={observed_csv_sha}",
)
df_raw = probe.load_flchain_dataframe(CSV_PATH, expected_sha256=probe.EXPECTED_CSV_SHA256)

assert set(probe.FLCHAIN_FEATURES).isdisjoint({"futime", "chapter", "death", "sample.yr", "sex", "Unnamed: 0"})
record_check(
    "zero_leakage_features", "all", "none",
    tuple(probe.FLCHAIN_FEATURES) == ("age", "sex_male", "kappa", "lambda", "flc.grp", "creatinine", "mgus"),
    float(len(probe.FLCHAIN_FEATURES)),
    "futime and chapter excluded; native NaNs preserved in creatinine",
)

censoring_df = probe.censoring_audit_table(df_raw, horizons=(1095, 1825, 2555))
censoring_df.to_csv(OUT_DIR / "censoring_audit.csv", index=False)
print(censoring_df[censoring_df["target"].isin(["horizon_1095d", "horizon_1825d"])].to_string(index=False))

       target  horizon_days   stratum  total_rows  censored_rows  censored_share  kept_rows  events  event_rate
horizon_1095d        1095.0      1995        1275             14        0.010980       1261      98    0.077716
horizon_1095d        1095.0      1996        3491             41        0.011744       3450     282    0.081739
horizon_1095d        1095.0      1997        1381             22        0.015930       1359     106    0.077999
horizon_1095d        1095.0      1998         687             17        0.024745        670      45    0.067164
horizon_1095d        1095.0      1999         350             10        0.028571        340      29    0.085294
horizon_1095d        1095.0      2000         245             11        0.044898        234      18    0.076923
horizon_1095d        1095.0      2001         175              7        0.040000        168      19    0.113095
horizon_1095d        1095.0      2002          48              2        0.041667         46       0    0

## 2. Target Cohorts & Deterministic Chronological Splits
Construct `1995-1997` trailing context (with `1997` stratified 50/25/25 into `context`, `ref_fit`, and `ref_eval`) and `1998-2003` future evaluation cohorts for `horizon_1095d` (primary), `horizon_1825d` (sensitivity), and `raw_death` (secondary).

In [3]:
TARGET_SPECS = [
    ("horizon_1095d", probe.PRIMARY_HORIZON_DAYS, True),
    ("horizon_1825d", probe.SENSITIVITY_HORIZON_DAYS, True),
    ("raw_death", None, False),
]
SENSITIVITY_TARGETS = [name for name, _, run_sens in TARGET_SPECS if run_sens]
assert "raw_death" not in SENSITIVITY_TARGETS

datasets = {}
for target_name, horizon_days, _ in TARGET_SPECS:
    kept_df, y = probe.build_flchain_target(df_raw, horizon_days)
    split = probe.split_flchain_chronological(kept_df, y)
    X = probe.extract_feature_matrix(kept_df)
    years = kept_df["sample.yr"].to_numpy(dtype=np.int64)

    ctx_s = set(kept_df.iloc[split["context_idx"]]["subject_id"])
    rfit_s = set(kept_df.iloc[split["ref_fit_idx"]]["subject_id"])
    reval_s = set(kept_df.iloc[split["ref_eval_idx"]]["subject_id"])
    fut_s = set(kept_df.iloc[split["future_idx"]]["subject_id"])
    disjoint = not (ctx_s & rfit_s or ctx_s & reval_s or ctx_s & fut_s or rfit_s & reval_s or rfit_s & fut_s or reval_s & fut_s)
    record_check(
        "split_disjointness", target_name, "none",
        disjoint and (len(ctx_s) + len(rfit_s) + len(reval_s) + len(fut_s) == len(kept_df)),
        float(len(kept_df)),
        f"context={len(ctx_s)} ({int(y[split['context_idx']].sum())} ev), "
        f"ref_fit={len(rfit_s)} ({int(y[split['ref_fit_idx']].sum())} ev), "
        f"ref_eval={len(reval_s)} ({int(y[split['ref_eval_idx']].sum())} ev), "
        f"future={len(fut_s)} ({int(y[split['future_idx']].sum())} ev)",
    )
    datasets[target_name] = {"df": kept_df, "X": X, "y": y, "years": years, "split": split}

## 3. Fit DR-TabPFN Context Models & Verify Parity / Replay / Gradients
Fit 5 DR-TabPFN (`single_fast`) models (`horizon_1095d` & `horizon_1825d` under consistent and historical scaling; `raw_death` under consistent scaling) and verify full-context retention, `matched_joint_execution` parity, head replay, finite-difference gradient accuracy, and the `1997_eval` `actual == held` identity control before saving analysis outputs.

In [4]:
RUN_SPECS = [
    ("horizon_1095d", "consistent", True, True),
    ("horizon_1095d", "historical", False, True),
    ("horizon_1825d", "consistent", True, True),
    ("horizon_1825d", "historical", False, True),
    ("raw_death", "consistent", True, False),
]

runs = {}
for target_name, scaling, use_ctx_scaler, run_held in RUN_SPECS:
    ds = datasets[target_name]
    X, y, years, split = ds["X"], ds["y"], ds["years"], ds["split"]
    ctx = split["context_idx"]

    fitted = probe.fit_dr_tabpfn_flchain(
        X[ctx], y[ctx], years[ctx],
        device=DEVICE, seed=probe.SPLIT_SEED, domain_origin=probe.DOMAIN_ORIGIN,
        use_context_scaler=use_ctx_scaler,
    )
    record_check(
        "context_no_silent_subsampling", target_name, scaling,
        fitted["subsample_rows"] == fitted["context_rows"],
        float(fitted["subsample_rows"]),
        f"context_rows={fitted['context_rows']}, subsample_rows={fitted['subsample_rows']}",
    )

    if target_name == "horizon_1095d" and scaling == "consistent":
        probe_idx = np.concatenate([split["ref_eval_idx"][:12], split["future_idx"][:20]])
        parity = probe.check_matched_joint_parity(fitted, X[probe_idx], years[probe_idx])
        record_check(
            "matched_joint_parity_32row", target_name, scaling,
            parity["passed"], max(parity["proba_max_abs_diff"], parity["emb_max_abs_diff"]),
            f"dp={parity['proba_max_abs_diff']:.2e}, dh={parity['emb_max_abs_diff']:.2e}",
        )

    eval_idx = np.concatenate([split["ref_eval_idx"], split["future_idx"]])
    ref_fit_cap = probe.capture_flchain_batches(fitted, X[split["ref_fit_idx"]], years[split["ref_fit_idx"]], batch_size=256)
    actual_cap = probe.capture_flchain_batches(fitted, X[eval_idx], years[eval_idx], batch_size=256)
    max_replay = max(ref_fit_cap["max_replay_error"], actual_cap["max_replay_error"])
    record_check(
        "same_call_head_replay", target_name, scaling,
        max_replay <= 1e-4, max_replay,
        f"max_replay_abs_err={max_replay:.2e}",
    )

    grads, fd_report = probe.compute_gradients_and_fd_check(
        actual_cap["continuation"], actual_cap["embeddings"], grad_batch_size=512, fd_rows=16
    )
    record_check(
        "layers11_finite_difference", target_name, scaling,
        fd_report["passed"], fd_report["max_absolute_error"],
        f"max_fd_err={fd_report['max_absolute_error']:.2e}",
    )

    held_cap = None
    if run_held:
        assert target_name in SENSITIVITY_TARGETS
        held_years = np.full(len(eval_idx), probe.REFERENCE_YEAR, dtype=np.int64)
        held_cap = probe.capture_flchain_batches(fitted, X[eval_idx], held_years, batch_size=256)
        n_reval = len(split["ref_eval_idx"])
        dp_1997 = float(np.max(np.abs(actual_cap["probabilities"][:n_reval] - held_cap["probabilities"][:n_reval])))
        dh_1997 = float(np.max(np.abs(actual_cap["embeddings"][:n_reval] - held_cap["embeddings"][:n_reval])))
        record_check(
            "ref_eval_actual_eq_held", target_name, scaling,
            (dp_1997 <= 1e-5) and (dh_1997 <= 1e-5), max(dp_1997, dh_1997),
            f"dp_1997={dp_1997:.2e}, dh_1997={dh_1997:.2e}",
        )

    ref_scale, zero_cols = probe.standardization_scale(ref_fit_cap["embeddings"])
    eval_pos = {int(orig_idx): pos for pos, orig_idx in enumerate(eval_idx)}
    runs[(target_name, scaling)] = {
        "run_held": run_held,
        "eval_idx": eval_idx,
        "eval_pos": eval_pos,
        "ref_fit_emb": ref_fit_cap["embeddings"],
        "ref_scale": ref_scale,
        "zero_scale_cols": zero_cols,
        "p_actual": actual_cap["probabilities"][:, 1],
        "emb_actual": actual_cap["embeddings"],
        "grad_actual": grads,
        "p_held": held_cap["probabilities"][:, 1] if held_cap is not None else None,
        "emb_held": held_cap["embeddings"] if held_cap is not None else None,
        "checkpoint_sha256": fitted["checkpoint_sha256"],
    }

    del fitted, ref_fit_cap, actual_cap, held_cap
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

checks_df = pd.DataFrame(checks)
checks_df.to_csv(OUT_DIR / "checks.csv", index=False)
print(checks_df.to_string(index=False))
if not checks_df["passed"].all():
    failed = checks_df.loc[~checks_df["passed"], "check_name"].tolist()
    raise RuntimeError(f"Pre-run checks failed: {failed}")

Initialized decoder for standard with (None, 10)  and nout 10


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{"absolute_error": [9.454786777496338e-06, 4.015862941741943e-06, 6.07222318649292e-06, 3.7178397178649902e-06, 3.769993782043457e-06, 1.7657876014709473e-06, 5.245208740234375e-06, 2.682209014892578e-07, 2.123415470123291e-06, 8.396804332733154e-06, 4.708766937255859e-06, 4.872679710388184e-06, 1.341104507446289e-06, 4.827976226806641e-06, 6.556510925292969e-07, 1.7508864402770996e-06], "analytical_derivatives": [0.10760775953531265, 0.11164740473031998, 0.10259758681058884, 0.1011975035071373, 0.10223011672496796, 0.09889277070760727, 0.09626525640487671, 0.10174092650413513, 0.10542187839746475, 0.10157620161771774, 0.0955733060836792, 0.10701309144496918, 0.09542539715766907, 0.09726768732070923, 0.10410183668136597, 0.0978606566786766], "atol": 1e-05, "central_derivatives": [0.10759830474853516, 0.11165142059326172, 0.10259151458740234, 0.10119378566741943, 0.10223388671875, 0.09889453649520874, 0.09626001119613647, 0.10174065828323364, 0.10541975498199463, 0.10156780481338501, 0.

{"absolute_error": [6.601214408874512e-06, 9.5367431640625e-06, 8.046627044677734e-07, 3.732740879058838e-06, 1.3113021850585938e-06, 1.0579824447631836e-06, 8.001923561096191e-06, 1.2539327144622803e-05, 1.0147690773010254e-05, 4.127621650695801e-06, 1.1116266250610352e-05, 1.4603137969970703e-06, 7.927417755126953e-06, 6.280839443206787e-06, 7.018446922302246e-06, 2.7418136596679688e-06], "analytical_derivatives": [0.1053893119096756, 0.1094961166381836, 0.10066112875938416, 0.10021255165338516, 0.10127109289169312, 0.09761558473110199, 0.09460841119289398, 0.10010091215372086, 0.10395823419094086, 0.10087306797504425, 0.09394505620002747, 0.1057833731174469, 0.09388130903244019, 0.0958341583609581, 0.10234607756137848, 0.09653294086456299], "atol": 1e-05, "central_derivatives": [0.10539591312408447, 0.10950565338134766, 0.10066032409667969, 0.1002088189125061, 0.10126978158950806, 0.09761452674865723, 0.09461641311645508, 0.10011345148086548, 0.10396838188171387, 0.10086894035339355

{"absolute_error": [4.656612873077393e-06, 5.736947059631348e-07, 5.364418029785156e-06, 1.3262033462524414e-06, 1.3209879398345947e-05, 5.02169132232666e-06, 3.6731362342834473e-06, 1.2665987014770508e-06, 4.7460198402404785e-06, 3.1962990760803223e-06, 7.778406143188477e-06, 1.2770295143127441e-05, 9.566545486450195e-06, 1.1608004570007324e-05, 1.4759600162506104e-05, 8.083879947662354e-06], "analytical_derivatives": [0.10696966201066971, 0.11015293747186661, 0.102120041847229, 0.10229365527629852, 0.10233690589666367, 0.09893979132175446, 0.095941923558712, 0.101676806807518, 0.10532168298959732, 0.10235735028982162, 0.09530004858970642, 0.10706271231174469, 0.09513559937477112, 0.09639294445514679, 0.10345564037561417, 0.09738143533468246], "atol": 1e-05, "central_derivatives": [0.10696500539779663, 0.11015236377716064, 0.10211467742919922, 0.10229498147964478, 0.10235011577606201, 0.09893476963043213, 0.09594559669494629, 0.10167807340621948, 0.10531693696975708, 0.102360546588897

{"absolute_error": [1.601874828338623e-05, 5.513429641723633e-07, 7.957220077514648e-06, 1.646578311920166e-06, 7.68899917602539e-06, 3.978610038757324e-06, 7.987022399902344e-06, 8.791685104370117e-06, 7.584691047668457e-06, 5.938112735748291e-06, 2.346932888031006e-06, 1.2218952178955078e-05, 4.887580871582031e-06, 8.687376976013184e-06, 5.684792995452881e-06, 8.344650268554688e-07], "analytical_derivatives": [0.10403133928775787, 0.10850484669208527, 0.10054060816764832, 0.1005159392952919, 0.10080760717391968, 0.09800393879413605, 0.09451115131378174, 0.10058596730232239, 0.10351674258708954, 0.1009121760725975, 0.09424027055501938, 0.10551244020462036, 0.09424495697021484, 0.09588773548603058, 0.10289820283651352, 0.09696269035339355], "atol": 1e-05, "central_derivatives": [0.10404735803604126, 0.1085042953491211, 0.10054856538772583, 0.10051429271697998, 0.1008152961730957, 0.0980079174041748, 0.09450316429138184, 0.10059475898742676, 0.10352432727813721, 0.10091811418533325, 0.0

{"absolute_error": [1.214444637298584e-06, 1.0423362255096436e-05, 3.4570693969726562e-06, 1.952052116394043e-06, 9.261071681976318e-06, 6.67572021484375e-06, 1.5430152416229248e-05, 2.3618340492248535e-06, 3.7103891372680664e-06, 7.152557373046875e-06, 2.8312206268310547e-07, 4.3213367462158203e-07, 1.3709068298339844e-06, 2.6971101760864258e-06, 6.034970283508301e-06, 2.7865171432495117e-06], "analytical_derivatives": [0.11497316509485245, 0.11470169574022293, 0.10276389122009277, 0.10404987633228302, 0.10405299812555313, 0.1003466248512268, 0.09768360108137131, 0.10278584808111191, 0.1061476618051529, 0.10263442993164062, 0.09578289091587067, 0.10789825022220612, 0.09561020135879517, 0.09744493663311005, 0.10401912033557892, 0.09865929186344147], "atol": 1e-05, "central_derivatives": [0.11497437953948975, 0.11471211910247803, 0.10276734828948975, 0.10405182838439941, 0.10406225919723511, 0.10033994913101196, 0.09766817092895508, 0.10278820991516113, 0.10614395141601562, 0.1026272773

## 4. Work 2 Last-Layer Geometry & Random-Direction TCAV Saturation
Compute raw and `ref_fit`-standardized embedding participation ratio, gradient coherence ($R$ and leading eigenvalue share), gradient PR, energy share in top-$k$ PCs, and random-direction TCAV interior share (`(0.1, 0.9)`, `2,000` draws) across reporting strata.

In [5]:
REPORT_STRATA = ("1997_eval", "1998", "1999", "2000", "2001", "2002-2003", "1998-2001", "1998-2003")
spec_rows, coh_rows, dim_rows, eng_rows, rand_rows = [], [], [], [], []

for (target_name, scaling), run in runs.items():
    ds = datasets[target_name]
    split = ds["split"]
    eval_pos = run["eval_pos"]
    for stratum in REPORT_STRATA:
        orig_idx = split["strata"][stratum]
        pos = np.array([eval_pos[int(i)] for i in orig_idx], dtype=np.int64)
        rng = geometry.build_rng(probe.DIRECTION_SEED, f"{target_name}:{scaling}:{stratum}", probe.REFERENCE_YEAR, 0)
        geom = probe.analyze_layer_geometry(
            run["emb_actual"][pos],
            run["grad_actual"][pos],
            run["p_actual"][pos],
            run["ref_scale"],
            rng=rng,
        )
        for chart, vals in geom["spectrum"].items():
            spec_rows.append({"target": target_name, "scaling": scaling, "stratum": stratum, "chart": chart, **vals})
        for (chart, gscale), vals in geom["coherence"].items():
            coh_rows.append({"target": target_name, "scaling": scaling, "stratum": stratum, "chart": chart, "gradient_scale": gscale, **vals})
        for (chart, gscale), vals in geom["dimension"].items():
            dim_rows.append({"target": target_name, "scaling": scaling, "stratum": stratum, "chart": chart, "gradient_scale": gscale, **vals})
        for (chart, gscale, k), val in geom["energy"].items():
            eng_rows.append({"target": target_name, "scaling": scaling, "stratum": stratum, "chart": chart, "gradient_scale": gscale, "k": k, "energy_share": val})
        for (chart, family), vals in geom["random"].items():
            rand_rows.append({"target": target_name, "scaling": scaling, "stratum": stratum, "chart": chart, "family": family, **vals})

spectrum_df = pd.DataFrame(spec_rows)
coherence_df = pd.DataFrame(coh_rows)
dimension_df = pd.DataFrame(dim_rows)
energy_df = pd.DataFrame(eng_rows)
random_df = pd.DataFrame(rand_rows)

spectrum_df.to_csv(OUT_DIR / "embedding_spectrum.csv", index=False)
coherence_df.to_csv(OUT_DIR / "gradient_coherence.csv", index=False)
dimension_df.to_csv(OUT_DIR / "gradient_dimension.csv", index=False)
energy_df.to_csv(OUT_DIR / "gradient_energy.csv", index=False)
random_df.to_csv(OUT_DIR / "random_direction_saturation.csv", index=False)

print(spectrum_df[(spectrum_df["target"] == "horizon_1095d") & (spectrum_df["scaling"] == "consistent")].to_string(index=False))

       target    scaling   stratum        chart  n_rows  embedding_pr  top1_var_share  top5_var_share  top10_var_share
horizon_1095d consistent 1997_eval          raw     340      2.505800        0.602800        0.926059         0.989054
horizon_1095d consistent 1997_eval standardized     340      4.217820        0.412194        0.871778         0.979784
horizon_1095d consistent      1998          raw     670      2.645845        0.578786        0.924632         0.987176
horizon_1095d consistent      1998 standardized     670      4.404106        0.389090        0.867613         0.975858
horizon_1095d consistent      1999          raw     340      2.709478        0.566224        0.925035         0.987437
horizon_1095d consistent      1999 standardized     340      4.460223        0.377920        0.869239         0.976685
horizon_1095d consistent      2000          raw     234      2.655888        0.574563        0.925059         0.986903
horizon_1095d consistent      2000 standardized 

## 5. Fixed-Patient `sample.yr` Sensitivity (`H = 1095` & `H = 1825` Only)
Compare identical patients evaluated under their actual `sample.yr` domain versus holding `sample.yr = 1997` fixed, reporting paired 1,000-draw bootstrap 95% CIs on predictive entropy (nats), mean probability, AUROC, and Brier score.

In [6]:
sens_rows = []
for (target_name, scaling), run in runs.items():
    if not run["run_held"]:
        continue
    assert target_name != "raw_death", "raw_death must never enter sample.yr sensitivity"
    ds = datasets[target_name]
    y = ds["y"]
    split = ds["split"]
    eval_pos = run["eval_pos"]
    for stratum in REPORT_STRATA:
        orig_idx = split["strata"][stratum]
        pos = np.array([eval_pos[int(i)] for i in orig_idx], dtype=np.int64)
        res = probe.paired_bootstrap_sensitivity(run["p_actual"][pos], run["p_held"][pos], y[orig_idx])
        flat = {
            "target": target_name, "scaling": scaling, "stratum": stratum,
            "n_rows": res["n_rows"], "n_events": res["n_events"],
        }
        for metric_key in (
            "mean_entropy_actual", "mean_entropy_held", "mean_entropy_diff",
            "mean_probability_actual", "mean_probability_held", "mean_probability_diff",
            "brier_actual", "brier_held", "brier_diff",
            "auroc_actual", "auroc_held", "auroc_diff",
        ):
            stat = res[metric_key]
            flat[metric_key] = stat["estimate"]
            flat[f"{metric_key}_ci_low"] = stat["ci_low"]
            flat[f"{metric_key}_ci_high"] = stat["ci_high"]
            if metric_key.startswith("auroc_"):
                flat[f"{metric_key}_valid_draws"] = stat["valid_draws"]
        sens_rows.append(flat)

sensitivity_df = pd.DataFrame(sens_rows)
assert "raw_death" not in set(sensitivity_df["target"])
sensitivity_df.to_csv(OUT_DIR / "sample_yr_sensitivity.csv", index=False)
cols_show = ["target", "scaling", "stratum", "n_rows", "n_events", "mean_entropy_actual", "mean_entropy_held", "mean_entropy_diff", "mean_entropy_diff_ci_low", "mean_entropy_diff_ci_high"]
print(sensitivity_df[cols_show].to_string(index=False))

       target    scaling   stratum  n_rows  n_events  mean_entropy_actual  mean_entropy_held  mean_entropy_diff  mean_entropy_diff_ci_low  mean_entropy_diff_ci_high
horizon_1095d consistent 1997_eval     340        26             0.173861           0.173861           0.000000                  0.000000                   0.000000
horizon_1095d consistent      1998     670        45             0.174642           0.173847           0.000795                  0.000191                   0.001322
horizon_1095d consistent      1999     340        29             0.166717           0.164801           0.001916                  0.001148                   0.002627
horizon_1095d consistent      2000     234        18             0.178048           0.174799           0.003249                  0.002317                   0.004246
horizon_1095d consistent      2001     168        19             0.182360           0.179318           0.003042                  0.001417                   0.004606
horizon_10

## 6. Model Performance & Simple CPU Reference Baselines
Compare DR-TabPFN against `LogisticRegression` (median imputation + missingness indicator fit on context rows only) and `HistGradientBoostingClassifier` across all cohorts and targets.

In [7]:
perf_rows = []
for (target_name, scaling), run in runs.items():
    ds = datasets[target_name]
    y = ds["y"]
    split = ds["split"]
    eval_pos = run["eval_pos"]
    modes = [("actual", run["p_actual"])]
    if run["p_held"] is not None:
        modes.append(("held_1997", run["p_held"]))
    for year_mode, p_vec in modes:
        for stratum in REPORT_STRATA:
            orig_idx = split["strata"][stratum]
            pos = np.array([eval_pos[int(i)] for i in orig_idx], dtype=np.int64)
            stats = probe.evaluate_probabilities_bootstrap(p_vec[pos], y[orig_idx])
            perf_rows.append({
                "target": target_name, "model": "DR-TabPFN", "scaling": scaling,
                "year_mode": year_mode, "stratum": stratum, **stats,
            })

for target_name, ds in datasets.items():
    X, y, split = ds["X"], ds["y"], ds["split"]
    ctx = split["context_idx"]
    eval_splits = {stratum: (X[split["strata"][stratum]], y[split["strata"][stratum]]) for stratum in REPORT_STRATA}
    base_df = probe.fit_and_score_cpu_baselines(X[ctx], y[ctx], eval_splits)
    for row in base_df.to_dict(orient="records"):
        perf_rows.append({"target": target_name, "scaling": "none", "year_mode": "none", **row})

perf_df = pd.DataFrame(perf_rows)
perf_df.to_csv(OUT_DIR / "model_performance.csv", index=False)
print(perf_df[(perf_df["target"] == "horizon_1095d") & (perf_df["stratum"].isin(["1997_eval", "1998-2003"]))][["model", "scaling", "year_mode", "stratum", "auroc", "auroc_ci_low", "auroc_ci_high", "brier", "mean_entropy"]].to_string(index=False))

               model    scaling year_mode   stratum    auroc  auroc_ci_low  auroc_ci_high    brier  mean_entropy
           DR-TabPFN consistent    actual 1997_eval 0.841499      0.737305       0.922385 0.051666      0.173861
           DR-TabPFN consistent    actual 1998-2003 0.864907      0.829982       0.896990 0.055072      0.164688
           DR-TabPFN consistent held_1997 1997_eval 0.841499      0.737305       0.922385 0.051666      0.173861
           DR-TabPFN consistent held_1997 1998-2003 0.862980      0.827894       0.895483 0.055053      0.161857
           DR-TabPFN historical    actual 1997_eval 0.838560      0.732935       0.921317 0.051673      0.176370
           DR-TabPFN historical    actual 1998-2003 0.864933      0.829858       0.895965 0.054892      0.172377
           DR-TabPFN historical held_1997 1997_eval 0.838560      0.732935       0.921317 0.051673      0.176370
           DR-TabPFN historical held_1997 1998-2003 0.863839      0.828629       0.896289 0.0550

## 7. SUS CKD Work 2 Comparison, Figures, Compact Archive, and Run Record
Summarize primary (`horizon_1095d`, `consistent`) last-layer metrics side by side with verified SUS CKD Work 2 benchmarks, render diagnostic figures, and persist compact splits/predictions and `run_record.json`.

In [8]:
def _extract_summary(stratum):
    s_raw = spectrum_df.query("target == 'horizon_1095d' and scaling == 'consistent' and stratum == @stratum and chart == 'raw'").iloc[0]
    s_std = spectrum_df.query("target == 'horizon_1095d' and scaling == 'consistent' and stratum == @stratum and chart == 'standardized'").iloc[0]
    c_raw = coherence_df.query("target == 'horizon_1095d' and scaling == 'consistent' and stratum == @stratum and chart == 'raw' and gradient_scale == 'probability'").iloc[0]
    d_raw = dimension_df.query("target == 'horizon_1095d' and scaling == 'consistent' and stratum == @stratum and chart == 'raw' and gradient_scale == 'probability'").iloc[0]
    d_std = dimension_df.query("target == 'horizon_1095d' and scaling == 'consistent' and stratum == @stratum and chart == 'standardized' and gradient_scale == 'probability'").iloc[0]
    r_iso = random_df.query("target == 'horizon_1095d' and scaling == 'consistent' and stratum == @stratum and chart == 'raw' and family == 'isotropic'").iloc[0]
    r_top = random_df.query("target == 'horizon_1095d' and scaling == 'consistent' and stratum == @stratum and chart == 'raw' and family == 'top'").iloc[0]
    r_cmp = random_df.query("target == 'horizon_1095d' and scaling == 'consistent' and stratum == @stratum and chart == 'raw' and family == 'complement'").iloc[0]
    return {
        "raw_embedding_pr": float(s_raw["embedding_pr"]),
        "std_embedding_pr": float(s_std["embedding_pr"]),
        "top5_var_raw": float(s_raw["top5_var_share"]),
        "top5_var_std": float(s_std["top5_var_share"]),
        "grad_resultant_length": float(c_raw["resultant_length"]),
        "grad_pr_raw_prob": float(d_raw["centered_pr"]),
        "grad_pr_std_prob": float(d_std["centered_pr"]),
        "random_tcav_interior_iso": float(r_iso["interior_share"]),
        "random_tcav_interior_top5": float(r_top["interior_share"]),
        "random_tcav_interior_comp": float(r_cmp["interior_share"]),
    }

sus_comp_df = probe.build_sus_comparison_table({
    "1997_eval": _extract_summary("1997_eval"),
    "1998-2003": _extract_summary("1998-2003"),
})
sus_comp_df.to_csv(OUT_DIR / "sus_comparison.csv", index=False)
print(sus_comp_df[["metric", "flchain_1997_eval", "flchain_1998_2003_pooled", "sus_ckd_work2_reference"]].to_string(index=False))

# Figure 1: Geometry summary across chronological strata
yr_strata = ["1997_eval", "1998", "1999", "2000", "2001", "2002-2003"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)
for chart, label in (("raw", "Raw"), ("standardized", "Std (ref_fit SD)")):
    sub = spectrum_df.query("target == 'horizon_1095d' and scaling == 'consistent' and chart == @chart").set_index("stratum").loc[yr_strata]
    axes[0].plot(yr_strata, sub["embedding_pr"], marker="o", label=label)
axes[0].axhspan(1.90, 3.21, alpha=0.15, color="C0", label="SUS CKD raw PR range")
axes[0].set_title("Last-Layer Embedding PR")
axes[0].legend(fontsize=8)

c_sub = coherence_df.query("target == 'horizon_1095d' and scaling == 'consistent' and chart == 'raw' and gradient_scale == 'probability'").set_index("stratum").loc[yr_strata]
d_sub = dimension_df.query("target == 'horizon_1095d' and scaling == 'consistent' and chart == 'raw' and gradient_scale == 'probability'").set_index("stratum").loc[yr_strata]
axes[1].plot(yr_strata, c_sub["resultant_length"], marker="o", label="Resultant R")
axes[1].axhspan(0.966, 0.992, alpha=0.15, color="C0", label="SUS CKD R range")
axes[1].set_ylim(0.0, 1.05)
axes[1].set_title(f"Head-Gradient Coherence R (PR={d_sub['centered_pr'].median():.2f})")
axes[1].legend(fontsize=8)

for fam in geometry.DIRECTION_FAMILIES:
    r_sub = random_df.query("target == 'horizon_1095d' and scaling == 'consistent' and chart == 'raw' and family == @fam").set_index("stratum").loc[yr_strata]
    axes[2].plot(yr_strata, r_sub["interior_share"], marker="o", label=fam)
axes[2].axhline(0.186, linestyle="--", color="gray", label="SUS CKD isotropic")
axes[2].set_title("Random-Direction Interior Share")
axes[2].set_ylim(0.0, 1.0)
axes[2].legend(fontsize=8)
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
fig.savefig(FIG_DIR / "geometry_summary.png", dpi=150)
plt.close(fig)

# Figure 2: Fixed-patient sample.yr predictive-entropy sensitivity
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
for ax, target_name, title in [(axes[0], "horizon_1095d", "3-Year Horizon (Primary)"), (axes[1], "horizon_1825d", "5-Year Horizon (Sensitivity)")]:
    for scaling in ("consistent", "historical"):
        sub = sensitivity_df.query("target == @target_name and scaling == @scaling").set_index("stratum").loc[yr_strata]
        yerr = np.vstack([sub["mean_entropy_diff"] - sub["mean_entropy_diff_ci_low"], sub["mean_entropy_diff_ci_high"] - sub["mean_entropy_diff"]])
        ax.errorbar(yr_strata, sub["mean_entropy_diff"], yerr=yerr, marker="o", capsize=3, label=f"{scaling} (actual - held)")
    ax.axhline(0.0, color="black", linestyle="--", linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel("Mean Entropy Diff (nats)")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=8)
fig.savefig(FIG_DIR / "sample_yr_entropy_sensitivity.png", dpi=150)
plt.close(fig)

# Figure 3: Model AUROC and Brier by stratum (horizon_1095d)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
model_series = [
    ("DR-TabPFN (consistent)", perf_df.query("target == 'horizon_1095d' and model == 'DR-TabPFN' and scaling == 'consistent' and year_mode == 'actual'")),
    ("DR-TabPFN (historical)", perf_df.query("target == 'horizon_1095d' and model == 'DR-TabPFN' and scaling == 'historical' and year_mode == 'actual'")),
    ("LogisticRegression", perf_df.query("target == 'horizon_1095d' and model == 'LogisticRegression'")),
    ("HistGradientBoosting", perf_df.query("target == 'horizon_1095d' and model == 'HistGradientBoosting'")),
]
for label, sub_df in model_series:
    sub = sub_df.set_index("stratum").loc[yr_strata]
    axes[0].plot(yr_strata, sub["auroc"], marker="o", label=label)
    axes[1].plot(yr_strata, sub["brier"], marker="o", label=label)
axes[0].set_title("3-Year AUROC by Cohort")
axes[1].set_title("3-Year Brier Score by Cohort")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=8)
fig.savefig(FIG_DIR / "model_performance_by_year.png", dpi=150)
plt.close(fig)

# Compact reusable archive for Work 8b
ds_3y = datasets["horizon_1095d"]
ds_5y = datasets["horizon_1825d"]
run_3y_con = runs[("horizon_1095d", "consistent")]
run_3y_hist = runs[("horizon_1095d", "historical")]
np.savez_compressed(
    OUT_DIR / "splits_and_predictions.npz",
    h1095_subject_id=ds_3y["df"]["subject_id"].to_numpy(dtype=np.int64),
    h1095_sample_yr=ds_3y["years"],
    h1095_y=ds_3y["y"],
    h1095_context_idx=ds_3y["split"]["context_idx"],
    h1095_ref_fit_idx=ds_3y["split"]["ref_fit_idx"],
    h1095_ref_eval_idx=ds_3y["split"]["ref_eval_idx"],
    h1095_future_idx=ds_3y["split"]["future_idx"],
    h1095_eval_idx=run_3y_con["eval_idx"],
    h1095_consistent_ref_scale=run_3y_con["ref_scale"],
    h1095_consistent_p_actual=run_3y_con["p_actual"],
    h1095_consistent_p_held=run_3y_con["p_held"],
    h1095_consistent_emb_actual=run_3y_con["emb_actual"],
    h1095_consistent_grad_actual=run_3y_con["grad_actual"],
    h1095_historical_p_actual=run_3y_hist["p_actual"],
    h1095_historical_p_held=run_3y_hist["p_held"],
    h1825_subject_id=ds_5y["df"]["subject_id"].to_numpy(dtype=np.int64),
    h1825_sample_yr=ds_5y["years"],
    h1825_y=ds_5y["y"],
    h1825_context_idx=ds_5y["split"]["context_idx"],
    h1825_ref_fit_idx=ds_5y["split"]["ref_fit_idx"],
    h1825_ref_eval_idx=ds_5y["split"]["ref_eval_idx"],
    h1825_future_idx=ds_5y["split"]["future_idx"],
)

git_head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
run_record = {
    "started_utc": STARTED_UTC,
    "finished_utc": datetime.now(timezone.utc).isoformat(),
    "runtime_seconds": round(time.monotonic() - T0, 2),
    "device": DEVICE,
    "gpu_name": GPU_NAME,
    "git_head": git_head,
    "versions": {"torch": torch.__version__, "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "csv_sha256": observed_csv_sha,
    "code_sha256": {
        "flchain_probe.py": probe.file_sha256("flchain_probe.py"),
        "flchain_probe.ipynb": probe.file_sha256("flchain_probe.ipynb"),
    },
    "zero_scale_cols": {f"{k[0]}:{k[1]}": v["zero_scale_cols"] for k, v in runs.items()},
    "all_checks_passed": bool(checks_df["passed"].all()),
    "n_checks": int(len(checks_df)),
}
(OUT_DIR / "run_record.json").write_text(json.dumps(run_record, indent=2))
print(f"Work 6b completed in {run_record['runtime_seconds']}s; all {run_record['n_checks']} checks passed.")

                                                metric  flchain_1997_eval  flchain_1998_2003_pooled  sus_ckd_work2_reference
                         Last-layer embedding PR (raw)           2.505800                  2.917207                    2.500
                Last-layer embedding PR (standardized)           4.217820                  4.953557                    4.080
                         Top-5 PC variance share (raw)           0.926059                  0.907536                    0.935
                Top-5 PC variance share (standardized)           0.871778                  0.841432                    0.875
           Head-gradient resultant length R (raw prob)           0.975438                  0.978200                    0.975
                  Head-gradient centered PR (raw prob)           1.173322                  1.139065                    1.080
                  Head-gradient centered PR (std prob)           1.078257                  1.059683                    1.030


Work 6b completed in 262.64s; all 25 checks passed.
